In [ ]:
# Select one config before running the remaining cells.
CONFIG_NAME = "Config_TBDT_Chem_dry_pretrained"

# Available choices:
# "Config_TBDT_Chem_dry_pretrained"
# "Config_TBDT_Chem_humid_pretrained"
# "Config_TBDT_Solvent_pretrained"
# "Config_TBDT_Bacteria_pretrained"


# Sensor classifier training

This notebook trains the CNN for the dataset selected in the first cell. The `training` folder is split
into model-training and validation partitions; `heldout_test` remains external
to fitting and checkpoint selection. The first epoch with maximum validation accuracy
is selected. Held-out inference runs only in the separate confusion-matrix cell
after training has restored the selected checkpoint. The confusion matrix includes
all held-out samples: RH 0%, 30%, and 50% for the humid configuration.
All code and data are local to this repository.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs import AVAILABLE_CONFIGS, load_config
from src import Data_pipeline, Run_analysis, Training_utils

Data_pipeline = importlib.reload(Data_pipeline)
Run_analysis = importlib.reload(Run_analysis)
Training_utils = importlib.reload(Training_utils)
CONFIG = load_config(CONFIG_NAME)
DATASET = CONFIG["DATASET"]
PREPROCESS = CONFIG["PREPROCESS"]
SELECTION = CONFIG["SELECTION"]
MODEL = CONFIG["MODEL"]
TRAINING = CONFIG["TRAINING"]
OUTPUT = CONFIG["OUTPUT"]
DATA_ROOT = Data_pipeline.resolve_data_root(DATASET["data_dir"])

print("Selected config:", CONFIG_NAME)
print("Dataset:", DATASET["name"])
print("Data directory:", DATA_ROOT)
print("Project root:", PROJECT_ROOT)
print("Compute device:", Training_utils.get_default_device())



## 1. Load and preprocess the dataset selected above


In [ ]:
data_bundle = Data_pipeline.prepare_data_bundle(
    data_root=DATA_ROOT,
    train_rh_values=SELECTION["train_rh_values"],
    heldout_test_rh_values=SELECTION["heldout_test_rh_values"],
    average_mode=PREPROCESS["average_mode"],
    cut_window=PREPROCESS["cut_window"],
    val_size=SELECTION["train_val_split"]["val_size"],
    random_state=SELECTION["train_val_split"]["random_state"],
)

print("Training input:", data_bundle["training"]["x"].shape)
print("Heldout-test input:", data_bundle["heldout_test"]["x"].shape)
print("Class labels:", data_bundle["gas_label_map"])


## 2. Train the CNN


In [ ]:
training_result = Training_utils.fit_model(
    data_bundle["dataset_train"],
    data_bundle["dataset_val"],
    num_classes=data_bundle["num_classes"],
    model_config=MODEL,
    training_config=TRAINING,
)

print("Checkpoint criterion: maximum validation accuracy (val_acc)")
print("Best epoch:", training_result["best_epoch"])


In [ ]:
# Evaluate once after training; RH-wise and pooled accuracies print before the figure.
heldout_evaluation = Run_analysis.evaluate_heldout(
    training_result["model"], data_bundle,
    device=training_result["device"], batch_size=TRAINING["batch_size"],
)
plt.show()


## 3. Save the reproducible run


In [ ]:
runs_root = PROJECT_ROOT / OUTPUT["models_dir"] / OUTPUT["runs_dirname"]
saved_run_dir = Training_utils.save_run_artifacts(
    training_result,
    config=CONFIG,
    gas_label_map=data_bundle["gas_label_map"],
    output_root=runs_root,
)
print("Saved run:", saved_run_dir)
